# M05-01 — Ranking por ventana

[← Anterior](01-teoria.ipynb) · [Siguiente →](03-lab-acumulados.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Obtener el top 10 de clientes por GMV y, por cada cliente, su top 3 productos.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M05-01-ranking-ventana.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo copiar desde este guion

El texto y el código que tienes que llevarte van en **celdas propias** (no dentro de un párrafo). Así se copian bien:

1. Clic en la celda (código o el Markdown corto del paso).
2. `Ctrl+A` (o `Cmd+A`) y `Ctrl+C` / `Cmd+C`. En Codespace también sale el icono de copiar en la barra de la celda.
3. Pega en **tu** notebook. La celda de código pégala como código; la de texto, como Markdown (`Esc` luego `M`).

No copies desde un recuadro gris a medias: usa la celda entera.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras. Puedes partir del texto corto del paso.
2. **Código** — pegas la celda de código, la **ejecutas** (`Shift+Enter`), **miras** la salida y, si no cuadra, la **mejoras**.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Top 10 clientes

**1.** En *tu* notebook, crea una celda Markdown. Copia **la celda de texto** que viene ahora (clic → `Ctrl+A` → `Ctrl+C`) y pégala. Luego déjala en tus palabras si quieres.

**2.** Crea debajo una celda de código. Copia **la celda de código** siguiente entera (igual: clic → seleccionar todo → copiar) y pégala.

**3.** Ejecuta en *tu* notebook (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** 10 filas, `rn` de 1 a 10, GMV decreciente. El nº 1 ronda **6000 €**.

**Por qué este paso.** Si no tienes customer_gmv, rehaz el groupBy de M04-03.


Sin partitionBy el ranking es de toda la compañía. row_number + where rn <= 10.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, row_number, sum as fsum
from pyspark.sql.window import Window

spark = get_spark("novashop-m05")
cust = spark.read.parquet(str(STAGING / "customer_gmv"))
w_global = Window.orderBy(col("gmv").desc())
top10 = cust.withColumn("rn", row_number().over(w_global)).where(col("rn") <= 10)
top10.orderBy("rn").show()


### Paso 2 — Top 3 productos por cliente

**1.** En *tu* notebook, crea una celda Markdown. Copia **la celda de texto** que viene ahora (clic → `Ctrl+A` → `Ctrl+C`) y pégala. Luego déjala en tus palabras si quieres.

**2.** Crea debajo una celda de código. Copia **la celda de código** siguiente entera (igual: clic → seleccionar todo → copiar) y pégala.

**3.** Ejecuta en *tu* notebook (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** Como mucho 3 filas por cliente; `rn` 1–3. `filas top3` ≤ 211 × 3.

**Por qué este paso.** Elige un customer_id con varios productos: sus rn empiezan en 1, no continúan el ranking global.


partitionBy reinicia el rn en cada cliente. Antes, groupBy cliente+producto (si no, la misma SKU se rankea varias veces).


In [ ]:
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
product_gmv = (
    fact.join(customers, "customer_id", "inner")
    .where(col("is_billable"))
    .groupBy("customer_id", "product_id")
    .agg(fsum("gmv_line").alias("gmv"))
)
w_prod = Window.partitionBy("customer_id").orderBy(col("gmv").desc())
top3 = product_gmv.withColumn("rn", row_number().over(w_prod)).where(col("rn") <= 3)
top_id = top10.select("customer_id").first()["customer_id"]
top3.where(col("customer_id") == top_id).show()
print("filas top3", top3.count())


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Elige un `customer_id` con varios productos y mira sus `rn`.
Empiezan en **1**. Escríbelo en Markdown (id + tres filas).


## Mejora — rank vs row_number

Fuerza un empate (o usa `rank` sobre gmv de clientes) y compara `rank` con `row_number`. Markdown: qué salta y qué no.

Si te atasca, copia la celda siguiente (clic → `Ctrl+A` → `Ctrl+C`) y pégala en *tu* notebook.


`row_number` nunca empata. `rank` repite y **salta** (1, 2, 2, 4). `dense_rank` no salta (1, 2, 2, 3).


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Un solo rn=1 en todo el fact | Olvidaste partitionBy | Añádelo para “por cliente” |
| Top 3 con 30 filas del mismo cliente | Rankeaste líneas | groupBy cliente+producto antes |
| Window sin orderBy | Ranking indefinido | Siempre ordena la métrica |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M05-02 acumulados](03-lab-acumulados.ipynb).
